# Phase 7 — Aspect-Based Sentiment Analysis (ABSA) Baseline

Example:

`The food was delicious but the service was very slow.`

Possible result:
- Food → Positive
- Service → Negative

This implementation is a practical, explainable baseline:
1. predefined restaurant aspects
2. sentence-level aspect detection
3. the trained Logistic Regression sentiment model

It can later be upgraded to a dedicated transformer-based ABSA model.


In [1]:
import os
import joblib
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import sent_tokenize

nltk.download("punkt")
try:
    nltk.download("punkt_tab")
except Exception:
    pass

print("Phase 7 imports loaded.")


Phase 7 imports loaded.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mahad\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mahad\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## 1. Load the trained model

In [2]:
model = joblib.load("../model/sentiment_model.joblib")
vectorizer = joblib.load("../model/tfidf_vectorizer.joblib")

print("Model:", type(model).__name__)
print("Classes:", model.classes_)


Model: LogisticRegression
Classes: ['Negative' 'Neutral' 'Positive']


## 2. Define restaurant aspects

These keywords can be expanded later.


In [3]:
aspect_keywords = {
    "Food": [
        "food", "dish", "meal", "taste", "flavor",
        "delicious", "menu", "cuisine"
    ],
    "Service": [
        "service", "waiter", "waitress", "staff",
        "server", "customer service"
    ],
    "Price": [
        "price", "prices", "cost", "expensive",
        "cheap", "value", "money", "affordable"
    ],
    "Ambience": [
        "ambience", "ambiance", "atmosphere",
        "decor", "interior", "music", "environment"
    ],
    "Location": [
        "location", "area", "parking", "place",
        "neighborhood"
    ],
    "Cleanliness": [
        "clean", "dirty", "cleanliness", "bathroom",
        "restroom"
    ],
    "Waiting Time": [
        "wait", "waiting", "slow", "quick",
        "fast", "minutes", "long time"
    ]
}

aspect_keywords


{'Food': ['food',
  'dish',
  'meal',
  'taste',
  'flavor',
  'delicious',
  'menu',
  'cuisine'],
 'Service': ['service',
  'waiter',
  'waitress',
  'staff',
  'server',
  'customer service'],
 'Price': ['price',
  'prices',
  'cost',
  'expensive',
  'cheap',
  'value',
  'money',
  'affordable'],
 'Ambience': ['ambience',
  'ambiance',
  'atmosphere',
  'decor',
  'interior',
  'music',
  'environment'],
 'Location': ['location', 'area', 'parking', 'place', 'neighborhood'],
 'Cleanliness': ['clean', 'dirty', 'cleanliness', 'bathroom', 'restroom'],
 'Waiting Time': ['wait',
  'waiting',
  'slow',
  'quick',
  'fast',
  'minutes',
  'long time']}

## 3. Sentence sentiment prediction

In [4]:
def predict_sentence_sentiment(sentence):
    vector = vectorizer.transform([sentence])
    prediction = model.predict(vector)[0]
    probabilities = model.predict_proba(vector)[0]
    confidence = float(np.max(probabilities))
    return prediction, confidence


## 4. Aspect analysis

In [5]:
def analyze_aspects(review):
    sentences = sent_tokenize(review)
    results = []

    for aspect, keywords in aspect_keywords.items():
        aspect_sentences = []

        for sentence in sentences:
            sentence_lower = sentence.lower()
            if any(keyword in sentence_lower for keyword in keywords):
                aspect_sentences.append(sentence)

        if not aspect_sentences:
            continue

        predictions = []
        for sentence in aspect_sentences:
            sentiment, confidence = predict_sentence_sentiment(sentence)
            predictions.append((sentiment, confidence, sentence))

        selected = max(predictions, key=lambda x: x[1])

        results.append({
            "aspect": aspect,
            "sentiment": selected[0],
            "confidence": round(selected[1], 4),
            "evidence": selected[2]
        })

    return pd.DataFrame(results)


## 5. Test ABSA

In [6]:
review = (
    "The food was delicious and the portions were large. "
    "The service was very slow and the waiter was not friendly. "
    "The restaurant was clean and the atmosphere was beautiful."
)

analyze_aspects(review)


,aspect,sentiment,confidence,evidence
0,Food,Positive,0.9381,The food was delicious and the portions were l...
1,Service,Negative,0.6334,The service was very slow and the waiter was n...
2,Ambience,Positive,0.7454,The restaurant was clean and the atmosphere wa...
3,Cleanliness,Positive,0.7454,The restaurant was clean and the atmosphere wa...
4,Waiting Time,Negative,0.6334,The service was very slow and the waiter was n...


## 6. Test a negative review

In [7]:
negative_review = (
    "The food was not good and the price was too expensive. "
    "The service was slow and the restaurant was dirty."
)

analyze_aspects(negative_review)


,aspect,sentiment,confidence,evidence
0,Food,Neutral,0.5553,The food was not good and the price was too ex...
1,Service,Negative,0.7978,The service was slow and the restaurant was di...
2,Price,Neutral,0.5553,The food was not good and the price was too ex...
3,Cleanliness,Negative,0.7978,The service was slow and the restaurant was di...
4,Waiting Time,Negative,0.7978,The service was slow and the restaurant was di...


## 7. Test a positive review

In [8]:
positive_review = (
    "The food was excellent. "
    "The staff was friendly and the service was great. "
    "The atmosphere was beautiful and the price was reasonable."
)

analyze_aspects(positive_review)


,aspect,sentiment,confidence,evidence
0,Food,Positive,0.9713,The food was excellent.
1,Service,Positive,0.9674,The staff was friendly and the service was great.
2,Price,Positive,0.7043,The atmosphere was beautiful and the price was...
3,Ambience,Positive,0.7043,The atmosphere was beautiful and the price was...


## 8. Analyze a dataset sample

In [9]:
df = pd.read_csv("../data/processed/yelp_reviews_nlp.csv")

sample_results = []

for idx, review in df["text"].head(100).items():
    aspect_df = analyze_aspects(review)

    for _, row in aspect_df.iterrows():
        sample_results.append({
            "review_index": idx,
            "aspect": row["aspect"],
            "sentiment": row["sentiment"],
            "confidence": row["confidence"],
            "evidence": row["evidence"]
        })

aspect_results_df = pd.DataFrame(sample_results)

print("Aspect records generated:", len(aspect_results_df))
aspect_results_df.head(10)


Aspect records generated: 258


,review_index,aspect,sentiment,confidence,evidence
0,0,Food,Positive,0.4964,"The food is good, but it takes a very long tim..."
1,0,Service,Positive,0.5391,"The waitstaff is very young, but usually pleas..."
2,0,Location,Negative,0.8024,I have been to it's other locations in NJ and ...
3,0,Waiting Time,Negative,0.8795,We have just had too many experiences where we...
4,1,Food,Positive,0.5755,Also has a menu with breakfast served all day ...
5,1,Service,Neutral,0.4967,"Friendly, attentive staff."
6,1,Location,Neutral,0.4509,Good place for a casual relaxed meal with no e...
7,1,Waiting Time,Positive,0.5755,Also has a menu with breakfast served all day ...
8,2,Food,Positive,0.9394,"Yummy, different, delicious."
9,3,Food,Positive,0.3945,Would like to see more menu options added such...


## 9. Aspect sentiment summary

In [10]:
if not aspect_results_df.empty:
    aspect_summary = pd.crosstab(
        aspect_results_df["aspect"],
        aspect_results_df["sentiment"]
    )
    display(aspect_summary)
else:
    print("No aspect records were generated.")


sentiment,Negative,Neutral,Positive
aspect,,,
Ambience,1,0,12
Cleanliness,2,0,4
Food,17,7,52
Location,8,5,37
Price,4,1,15
Service,13,5,29
Waiting Time,11,3,32


## Phase 7 conclusion

The ABSA baseline returns:
- aspect
- sentiment
- confidence
- evidence sentence

The same logic will later be exposed through FastAPI.
